# Student Academic Performance Analysis

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

## Dataset disclosure
The included student dataset is synthetic demonstration data created for learning. It does not represent actual students or a real school. It is useful for practising EDA, but do not make claims about real student populations from it.

## Objectives
- Inspect and clean student records.
- Summarise academic scores.
- Explore associations between study time, attendance, tutoring and scores.
- Communicate patterns with visualisations.

**Important:** Correlation or association does not prove causation.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'student_performance_demo.csv'
OUT = ROOT / 'visualizations'
OUT.mkdir(exist_ok=True)
df_raw = pd.read_csv(DATA)
df_raw.head()


## 1. Data inspection


In [ ]:
print('Shape:', df_raw.shape)
df_raw.info()
display(df_raw.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', df_raw.duplicated().sum())


## 2. Data cleaning and preparation
Remove duplicate rows, fill numeric missing values with medians and categorical missing values with modes. Explain why you choose these methods in a real project report.


In [ ]:
df = df_raw.drop_duplicates().copy()
numeric_cols = ['study_time_hours_per_week','attendance_percent','absences','previous_score','final_score']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
for col in ['tutoring','internet_access','parent_education','gender']:
    df[col] = df[col].fillna(df[col].mode()[0])
df['performance_band'] = pd.cut(df['final_score'], bins=[-0.1,49.9,69.9,84.9,100], labels=['Needs support','Satisfactory','Good','Excellent'])
print('Cleaned shape:', df.shape)
df.head()


## 3. Descriptive analysis


In [ ]:
display(df['final_score'].describe().to_frame('final_score'))
display(df['performance_band'].value_counts().to_frame('student_count'))
display(df.groupby('gender')['final_score'].agg(['mean','median','count']).round(2))


## 4. Visualisations


In [ ]:
plt.figure(figsize=(9,5))
plt.hist(df['final_score'], bins=20)
plt.title('Distribution of Final Scores'); plt.xlabel('Final score')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.scatter(df['study_time_hours_per_week'], df['final_score'], alpha=0.35)
plt.title('Study Time and Final Score'); plt.xlabel('Study time (hours per week)'); plt.ylabel('Final score')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8,5))
df.boxplot(column='final_score', by='tutoring')
plt.title('Final Scores by Tutoring Status'); plt.xlabel('Receives tutoring'); plt.ylabel('Final score')
plt.tight_layout(); plt.show()


In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(8,6))
plt.imshow(corr.values, vmin=-1, vmax=1); plt.xticks(range(len(corr.columns)), corr.columns, rotation=35, ha='right'); plt.yticks(range(len(corr.index)), corr.index); plt.colorbar()
plt.title('Correlation Between Numeric Student Variables')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,5))
order = ['Needs support','Satisfactory','Good','Excellent']
df['performance_band'].value_counts().reindex(order).plot(kind='bar')
plt.title('Number of Students in Each Performance Band')
plt.xlabel('Performance band'); plt.ylabel('Number of students')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()


## 5. Interpretation and limitations
Write findings based on the charts and summary tables after running the notebook. Compare group averages carefully, identify visible relationships, and avoid claiming that study time or tutoring caused a particular score.

**Limitations:** The dataset is synthetic and does not represent actual students. The patterns are intended to exercise analysis methods only. A real study would need a properly sourced dataset, ethical handling of student information, and context about how the data was collected.
